# Lab - Unit 8: identify the real car

**Runs on your machine with CARLA 0.9.12 and `ROAR_PY`.**

On racekit, Unit 8 measured three things about the carla-like car from logs: the longitudinal map (a_T 5.17 m/s², a_B 8.54 m/s², drag c 3.08e-4 1/m, one tick of delay), the steering gain (65.7°, where Unit 7 assumed 70°), and the grip of each section with one-lap **limit probes**.
Planning with a fraction f of the measured grip, one f tuned on the training family transferred to an unseen track.
**In this lab you measure the same things on CARLA's car and Monza**, and then race a short, fixed ladder of fractions, accepting only a setting that was clean on both of its races.

What you do, in order:

1. record two runs of your Unit 7 agent: one race, and one steady lap;
2. fit the longitudinal map and the steering gain on **your** logs, and compare them with racekit's;
3. run the six grip probes on CARLA, one section each, and turn them into six μ̂·g values with the lesson's rules;
4. predict with racekit first (the asserted cell of §5), then race the decision procedure of §6;
5. never commit your results.

How to use it:

- The cells tagged `no-exec` are for **your machine** (the Windows PC you set up in the Unit 1 lab).
  Their shell lines are shown with `!` only so the notebook stays valid.
- The analysis cells run anywhere.
  In CI they run against a racekit-made sample recording (the starter on `monza-like`, with no section file, no respawn and no hard braking), so the book always proves they work; on your machine they run against your own recordings, **one at a time**.

> **Never commit your results.**
> This repository is public.
> Your recordings (the telemetry CSVs **and** their `.sections.txt` sidecars), screenshots and logs stay on your own machine, **outside the repository**
> (the Unit 1 lab uses `%USERPROFILE%\roar-runs\`).
> The repository ignores data files under `labs/`, and a CI test fails if anything other than a notebook
> or a README is ever tracked there, but the first line of defence is not putting them here at all.

In [ ]:
import os

TELEMETRY_PATH = os.environ.get("ROAR_LAB_CSV", "telemetry.csv")

The parameters cell above is the **only** place a recording's file name appears.
Set `ROAR_LAB_CSV` to one recording's full path (outside the repository) before you start Jupyter; the recorder writes to the same variable, and the section sidecar's name is that path with `.sections.txt` added.
This lab makes **eight** recordings, so give each its own name (the commands in §2 do) and point `ROAR_LAB_CSV` at the one you are analysing.

## 1. The five steps

The Unit 1 lab (`units/unit-01-the-race-and-baseline/labs/real-sim.ipynb`) has every detail; this lab only changes what you race and what you record.

1. **Start the Monza v1.1 executable** (CARLA 0.9.12) and wait for the map to load (Unit 1 lab, §1 for the pinned versions and §3 for the start).
2. **Put the submission in the forked competition folder**: start from the file you raced in the Unit 7 lab, `agents/unit-07.py` with the Unit 1 recorder and its three wiring lines, Unit 5's one-line `offset` change and Unit 7's section sidecar (Unit 7 lab, §2), all unchanged.
   Each run below changes only a few constants at the top of that file (§2's table).
   Before you go to CARLA, race the edited file on racekit and run `check_submission` on it, as the Unit 1 lab says.
3. **Run `competition_runner.py`** from `ROAR_Competition\competition_code`, with `ROAR_LAB_CSV` set to a new file outside the repository.
   Note the time and the collision count the runner reports, and whether the race finished.
4. **Export the telemetry to CSV**: the recorder does it as the car drives, one row per tick, to `ROAR_LAB_CSV`, and writes the sidecar once, when the race starts.
   It writes every row as it goes, so a run you stop early (with Ctrl+C in the prompt) still leaves a complete recording up to that tick.
5. **Analyse it**: start Jupyter with `ROAR_LAB_CSV` pointing at the recording and run §3 onwards of this notebook.

## 2. The runs

| run | file | constants changed at the top of the file | how long |
|---|---|---|---|
| **R1**, the race | your Unit 7 lab file | none | the runner's full race |
| **R2**, the steady lap | your Unit 7 lab file | `TARGET_SPEED = 15.0`, `SPEED_MODE = "flat"` | **one lap**, then stop the runner (about 6.5 minutes at 15 m/s) |
| **P0** … **P5**, the probes | your Unit 7 lab file | `A_LAT = 3.5`, and `GRIP` = 1.0 everywhere except **3.5 in section s** (P2: `GRIP = (1.0, 1.0, 3.5, 1.0, 1.0, 1.0)`) | stop at the **first respawn**, or after **3 minutes** if there is none |
| the decision races | §6 says which file and which constants | §6 | the runner's full race |

**R1** is for the longitudinal map: a real race brakes hard and accelerates hard, so every column of the fit is excited.
**R2** is for the steering gain: at a flat 15 m/s the steer holds still through Monza's long bends, which is what the steady-row rule needs.
**The probes** are the lesson's: your own agent, gentle everywhere (`A_LAT = 3.5`) except one section, where the planner asks for 3.5 × 3.5 = 12.25 m/s² of cornering, more than any section of racekit's tracks gave.
The car arrives too fast in that section's corners and slides; the slide, at the limit, is in the recorded positions.
Stop at the first respawn: after it the recorder starts the race again from the start (Unit 1 lab), and one slide is what you came for.
A probe that runs 3 minutes without a respawn never slid: its number will be a **lower bound**.
CARLA's sections are your agent's own `self.section`, written to the sidecar (Unit 7 lab, §2), so the analysis knows which rows were in the pushed section.

On CARLA a probe **will** hit the wall.
That is its job; nothing else in the book does it on purpose, so run the probes only on your own lab machine, after the race and the steady lap.

In [ ]:
# Anaconda Prompt, on the Windows machine, from ROAR_Competition\competition_code. Not run in CI.
# Type each line WITHOUT the leading "!" (it is there only so the notebook stays valid).
# 1. Your Unit 7 lab file is submission.py; edit its constants for each run as section 2's table says.
# 2. R1, the race (no constants changed)
!set "ROAR_LAB_CSV=%USERPROFILE%\roar-runs\unit-08-r1-race.csv" && python competition_runner.py
# 3. R2, the steady lap (TARGET_SPEED = 15.0, SPEED_MODE = "flat"); Ctrl+C after one lap
!set "ROAR_LAB_CSV=%USERPROFILE%\roar-runs\unit-08-r2-steady.csv" && python competition_runner.py
# 4. P0 ... P5, one probe per section (A_LAT = 3.5, GRIP[s] = 3.5); Ctrl+C at the first respawn or after 3 minutes
!set "ROAR_LAB_CSV=%USERPROFILE%\roar-runs\unit-08-p0.csv" && python competition_runner.py
!set "ROAR_LAB_CSV=%USERPROFILE%\roar-runs\unit-08-p1.csv" && python competition_runner.py
# ... and so on, up to unit-08-p5
# 5. Analyse one recording: start Jupyter with the variable pointing at it (from your clone of this repository)
!set "ROAR_LAB_CSV=%USERPROFILE%\roar-runs\unit-08-r1-race.csv" && jupyter lab

## 3. Fit the car on your recording

Point `ROAR_LAB_CSV` at **R1** for the longitudinal map and at **R2** for the steering gain; the cell below runs both fits on whatever recording it is given, and says which one to trust.
The fits are the lesson's, on the telemetry's columns: the speed is `v`, the yaw rate is the wrapped heading change per tick, and a row counts only when it, and every row the fit pairs it with, is a driving row (`held == 0`) one tick from its neighbour.

- **The longitudinal map**: dv/dt = a_T·throttle − a_B·brake − c·v|v| by least squares, for lags 0–3; the lag with the smallest residual RMS wins.
- **The steering gain**: rows where the steer logged one tick earlier has been still (standard deviation below 0.002 over 11 ticks), |steer| > 0.005 and v > 5 m/s; the wheel angle atan(r·L / v) is fitted as MAX_STEER · (−steer).
  With 20 rows or fewer there is no gain.

It also reports whether the recording is **clean** by Unit 7's rule (the runner reports a finished race, no respawn, max |cross-track error| ≤ 2 m on the driving rows without the stale first row after a hold): the decision races of §6 are judged by it.
Set `RACE_FINISHED` to what the runner reported (Unit 1 lab, §4, explains why the recorder cannot know).

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
from racekit.telemetry import read_csv

RACE_FINISHED = True  # set False if the runner reported that the race did not finish
CONTROL_DT = 0.05  # s: one row per control tick
WHEELBASE = 2.875  # m, L (Unit 3)
CLEAN_OFFSET = 2.0  # m: the clean rule's limit
MIN_ROWS = 20  # 20 rows or fewer: no estimate


def one_tick(t, first, second):
    # rows first -> second are one control tick apart
    return np.abs(t[second] - t[first] - CONTROL_DT) < 1e-3


def longitudinal_fit(rows, lag):
    # rows: a recording's columns (t, v, throttle, brake, held). Returns ((a_T, a_B, c), residual RMS, equations).
    t, v, held = rows["t"], rows["v"], rows["held"]
    k = np.arange(lag, len(t) - 1)
    keep = one_tick(t, k, k + 1) & (held[k] == 0) & (held[k + 1] == 0)
    for back in range(1, lag + 1):
        keep = keep & one_tick(t, k - back, k - back + 1) & (held[k - back] == 0)
    k = k[keep]
    mean = (v[k] + v[k + 1]) / 2.0
    a = np.column_stack([rows["throttle"][k - lag], -rows["brake"][k - lag], -mean * np.abs(mean)])
    b = (v[k + 1] - v[k]) / CONTROL_DT
    c = np.linalg.lstsq(a, b, rcond=None)[0]
    return c, float(np.sqrt(np.mean((a @ c - b) ** 2))), len(k)


def best_longitudinal_fit(rows, lags=(0, 1, 2, 3)):
    fits = {lag: longitudinal_fit(rows, lag) for lag in lags}
    lag = min(fits, key=lambda key: fits[key][1])
    return lag, fits


def steering_gain(rows, lag=1, stillness_limit=0.002):
    # (steady rows, MAX_STEER in degrees or None): the yaw change k -> k + 1 against the steer logged at k - lag.
    t, yaw, v, held = rows["t"], rows["yaw"], rows["v"], rows["held"]
    k = np.arange(lag + 10, len(t) - 1)
    turn = (yaw[k + 1] - yaw[k] + math.pi) % (2.0 * math.pi) - math.pi
    rate = turn / CONTROL_DT
    speed = (v[k] + v[k + 1]) / 2.0
    applied = rows["steer"][k - lag]
    stillness = np.array([np.std(rows["steer"][j - lag - 10:j - lag + 1]) for j in k])
    driving = np.array([bool(np.max(held[j - lag - 10:j + 2]) == 0) for j in k]) & one_tick(t, k, k + 1)
    steady = driving & (speed > 5.0) & (np.abs(applied) > 0.005) & (stillness < stillness_limit)
    count = int(np.sum(steady))
    if count <= MIN_ROWS:
        return count, None
    wheel_angle = np.array([math.atan2(r_k * WHEELBASE, v_k) for r_k, v_k in zip(rate[steady], speed[steady])])
    gain = np.linalg.lstsq(np.column_stack([-applied[steady]]), wheel_angle, rcond=None)[0][0]
    return count, math.degrees(gain)


def clean_check(telemetry):
    # Unit 7 lab's rule: no hold run, and max |offset| <= 2 m on driving rows without the stale row after a hold.
    held = telemetry["held"] == 1
    keep = ~held
    stale = [row + 1 for row in np.nonzero(held)[0] if row + 1 < len(held) and not held[row + 1]]
    keep[np.array(stale, dtype=int)] = False
    respawns = len([row for row in np.nonzero(held)[0] if row == 0 or not held[row - 1]])
    worst = float(np.max(np.abs(telemetry["offset"][keep]))) if np.sum(keep) else float("nan")
    return RACE_FINISHED and respawns == 0 and worst <= CLEAN_OFFSET, respawns, worst


telemetry, meta = read_csv(TELEMETRY_PATH)
n = meta["waypoints_per_lap"]
is_clean, respawns, worst = clean_check(telemetry)
print(f"your recording: {len(telemetry)} rows, {respawns} respawns, max |cross-track error| {worst:.3f} m")
print(f"clean by Unit 7's rule (finished, no respawn, max |e| <= {CLEAN_OFFSET} m): {is_clean}")

my_lag, my_fits = best_longitudinal_fit(telemetry)
for lag, (c, residual, count) in my_fits.items():
    print(f"lag {lag}: a_T {c[0]:6.3f}, a_B {c[1]:6.3f}, c {c[2]:.3e}   RMS {residual:.3f} m/s^2 ({count} rows)")
braking_rows = int(np.sum(telemetry["brake"] > 0.1))
print(f"the residual chooses lag {my_lag}; rows braking harder than 0.1: {braking_rows}")
if braking_rows < 200:
    print("too little braking to trust a_B: fit the longitudinal map on R1, the race")

my_gain_rows, my_gain = steering_gain(telemetry)
if my_gain is None:
    print(f"steering gain: no estimate ({my_gain_rows} steady rows): point ROAR_LAB_CSV at R2, the steady lap")
else:
    print(f"steering gain: {my_gain:.2f} deg from {my_gain_rows} steady rows (Unit 7 assumed 70 deg)")

fig, ax = plt.subplots(figsize=(7, 3.5))
lags = sorted(my_fits)
ax.plot(lags, [my_fits[lag][1] for lag in lags], "o-")
ax.set_xlabel("lag (ticks)")
ax.set_ylabel("residual RMS (m/s^2)")
ax.set_title("your recording: the residual chooses the lag")
plt.show()

Write down, from **R1**: the lag and the three coefficients; and from **R2**: the steering gain.
Your gain is the one number from this section the submission uses (§6); the longitudinal map is for comparison (the submission does not drive with it until Unit 10).

## 4. The grip probes, one recording at a time

Point `ROAR_LAB_CSV` at one probe recording, set `PROBE_SECTION` to the section it pushed, and run the cell.
It applies the lesson's rule to the recording:

- |a| is the second difference of the recorded positions, (p_{k+1} − 2p_k + p_{k−1}) / 0.05²;
- a row counts when it and its two neighbours are driving rows (`held == 0`) one tick apart, both steps are shorter than 5 m (a respawn teleports the car) and v > 5 m/s;
- a row's section is the sidecar's `sections[progress % n]` (Unit 7 lab, §2; without a sidecar, six equal slices of the progress count, which the cell says);
- the number of a section is the 99th percentile of its rows' |a|.

The **pushed section's** number is what the probe measures, and the rules say what it is:

- a probe that **respawned** (it slid and crashed) gives an **estimate**;
- a probe that **never respawned** gives a **lower bound** (the grip is at least this);
- a section with **20 qualifying rows or fewer** gives **no estimate**.

Log the pushed section's line in `PROBES` (§6), one probe at a time.

In [ ]:
PROBE_SECTION = 0  # the section this recording's probe pushed (GRIP[s] = 3.5)
SECTIONS = 6


def load_sections(csv_path, n):
    # The Unit 7 recorder's sidecar, <csv_path>.sections.txt: line k is the section of progress k (mod n).
    # Without one, six equal slices of the progress count; returns (sections, True when from the sidecar).
    try:
        with open(f"{csv_path}.sections.txt") as handle:
            sections = [int(handle.readline()) for _ in range(n)]
    except FileNotFoundError:
        return [min(k * SECTIONS // n, SECTIONS - 1) for k in range(n)], False
    return sections, True


def section_accel(rows, sections):
    # Per section: (qualifying rows, the 99th percentile of |a| or None); sections in progress order.
    t, x, y, held = rows["t"], rows["x"], rows["y"], rows["held"]
    accel = np.hypot((x[2:] - 2.0 * x[1:-1] + x[:-2]) / CONTROL_DT ** 2,
                     (y[2:] - 2.0 * y[1:-1] + y[:-2]) / CONTROL_DT ** 2)
    step = np.hypot(np.diff(x), np.diff(y))
    middle = np.arange(1, len(t) - 1)
    ok = (held[:-2] == 0) & (held[1:-1] == 0) & (held[2:] == 0)
    ok = ok & (step[:-1] < 5.0) & (step[1:] < 5.0) & (rows["v"][1:-1] > 5.0)
    ok = ok & one_tick(t, middle - 1, middle) & one_tick(t, middle, middle + 1)
    section = np.array(sections)[np.asarray(rows["progress"][1:-1]) % len(sections)]
    table = []
    for s in range(SECTIONS):
        values = accel[ok & (section == s)]
        table.append((len(values), float(np.percentile(values, 99.0)) if len(values) > MIN_ROWS else None))
    return table


def respawned(rows):
    # a hold row, or progress going backwards, means the car was put back on the track
    return bool(np.sum(rows["held"] == 1) > 0 or np.min(np.diff(rows["progress"])) < 0)


def verdict(rows_count, value, slid):
    if rows_count <= MIN_ROWS or value is None:
        return "no estimate"
    return "estimate" if slid else "lower bound"


probe_sections, from_sidecar = load_sections(TELEMETRY_PATH, n)
if not from_sidecar:
    print("no .sections.txt sidecar next to this recording: using six equal slices of the progress count")
probe_table = section_accel(telemetry, probe_sections)
slid = respawned(telemetry)
print(f"this recording respawned: {slid}")
print(f"{'section':>8}{'rows':>7}{'p99 |a| (m/s^2)':>17}")
for s, (rows_count, value) in enumerate(probe_table):
    shown = "-" if value is None else f"{value:.2f}"
    marker = "   <- pushed" if s == PROBE_SECTION else ""
    print(f"{s:8d}{rows_count:7d}{shown:>17}{marker}")
pushed_rows, pushed_value = probe_table[PROBE_SECTION]
print(f"log this line in PROBES: {PROBE_SECTION}: ({pushed_rows}, "
      f"{None if pushed_value is None else round(pushed_value, 2)}, {slid}),"
      f"   -> {verdict(pushed_rows, pushed_value, slid)}")

## 5. racekit's prediction

Before any decision race, see what the same procedure gives on racekit.
This cell runs the **same** functions on racekit, and **asserts** the results, so CI proves the numbers you compare against:

- the six probes of `test-circuit` (your Unit 7 agent, `A_LAT = 3.5`, `GRIP[s] = 3.5`, one lap, 300 s, seed 0) reproduce `agents/unit-08.py`'s `MU_G`, and every one of them crashed (plan `roar-008` U8.2);
- the fits of racekit's committed logs (`racekit.sysid_data`) give lag 1, a_T 5.17, a_B 8.54, c 3.08e-4 and a gain of 65.71° (U8.1).

racekit's progress counter starts at the waypoint *after* the start waypoint, so its section map in progress order is the agent's map shifted by one: line k is the section of waypoint k + 1.
The committed logs have the velocity vector, not `v`, and no `held` column, so the cell builds those two columns for them.
racekit is not CARLA: compare the **pattern** (how much the grip differs between sections, how far your gain is from 70°), not the second decimal.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp

from racekit import write_variant
from racekit.cars import carla_like
from racekit.runner import run_race
from racekit.sysid_data import LOGS, load_log
from racekit.tracks import grip_sections, load_track

AGENT = Path("..") / ".." / ".." / "projects" / "racekit" / "agents" / "unit-07.py"
SHIPPED_MU_G = (7.89, 9.72, 8.65, 7.87, 8.30, 8.14)  # agents/unit-08.py's MU_G (test-circuit), plan roar-008 U8.2
workdir = Path(mkdtemp(prefix="roar-lab-08-"))
track = load_track("test-circuit")
grip = grip_sections(track)
model_sections = [int(grip[(k + 1) % len(grip)]) for k in range(len(grip))]  # racekit's progress order
model_mu_g = []
for s in range(SECTIONS):
    knobs = [1.0] * SECTIONS
    knobs[s] = 3.5
    source = write_variant(AGENT, workdir / f"probe-{s}.py", {"A_LAT": "3.5", "GRIP": repr(tuple(knobs))})
    probe_race = run_race(source, track, car=carla_like(noise=True), seed=0, max_seconds=300.0, laps=1)
    rows_count, value = section_accel(probe_race.telemetry, model_sections)[s]
    assert respawned(probe_race.telemetry) and verdict(rows_count, value, True) == "estimate", (s, rows_count)
    model_mu_g.append(round(value, 2))
assert tuple(model_mu_g) == SHIPPED_MU_G, model_mu_g


def log_rows(log):
    # a committed sensor log as the recording's columns: v from the velocity vector, held all 0
    return {"t": log["t"], "x": log["x"], "y": log["y"], "yaw": log["yaw"], "v": np.hypot(log["vx"], log["vy"]),
            "throttle": log["throttle"], "steer": log["steer"], "brake": log["brake"], "held": np.zeros(len(log))}


race_log_name, steady_log_name = LOGS
model_lag, model_fits = best_longitudinal_fit(log_rows(load_log(race_log_name)[0]))
model_map = model_fits[model_lag][0]
model_gain_rows, model_gain = steering_gain(log_rows(load_log(steady_log_name)[0]))
assert model_lag == 1
assert abs(model_map[0] - 5.174) < 0.001 and abs(model_map[1] - 8.539) < 0.001 and abs(model_map[2] - 3.081e-4) < 1e-7
assert abs(model_gain - 65.71) < 0.005, model_gain

print(f"racekit, test-circuit, the six probes: mu * g = {tuple(model_mu_g)} m/s^2 (every probe crashed)")
print(f"racekit, the committed logs: lag {model_lag}, a_T {model_map[0]:.2f}, a_B {model_map[1]:.2f}, "
      f"c {model_map[2]:.2e}; steering gain {model_gain:.2f} deg ({model_gain_rows} steady rows)")
print()
print(f"{'':24}{'you (this recording)':>22}{'racekit':>12}")
print(f"{'lag (ticks)':24}{my_lag:>22d}{model_lag:>12d}")
for i, name in enumerate(("a_T (m/s^2)", "a_B (m/s^2)", "c (1/m)")):
    print(f"{name:24}{my_fits[my_lag][0][i]:>22.4g}{model_map[i]:>12.4g}")
shown_gain = "-" if my_gain is None else f"{my_gain:.2f}"
print(f"{'steering gain (deg)':24}{shown_gain:>22}{model_gain:>12.2f}")

## 6. The decision procedure

**Step 3's result: CARLA's grip, from your six probes.**
Log every probe's pushed-section line from §4 in `PROBES`.
A section with an **estimate** keeps it; a section with a **lower bound** keeps the bound (the grip is at least that, so planning with it can only be cautious, as racekit's `monza-like` section 0 is); a section with **no estimate** takes the **lowest estimate** on the lap.
If **no** section has an estimate (no probe respawned), the grip is **not identified**, and the procedure takes the no-estimate path below.

**The steering gain**: ship 66° (racekit's measurement, rounded), or your own R2 gain rounded to a whole degree (`MY_GAIN_DEG`).
If your gain is far from 66°, trust yours: it was measured on the car you are racing.

**Step 4: the decision races.**
Run §5 first: it is the prediction, and a surprise there (a gain far from racekit's, a probe that never slid) is worth understanding before you race.
Then:

- **With the grip identified**: race `agents/unit-08.py` (with the recorder, Unit 5's change and the sidecar pasted in, as for your Unit 7 lab file), with `MAX_STEER = math.radians(<your gain>)`, `MU_G` = the six values the cell below prints, and `MU_FRACTION` = **0.54** first: one grid step inside the racekit family's frozen 0.56, because CARLA is a track the family never saw.
  Race it **twice**, and stop at the first race that is not clean (§3's rule).
  Clean twice: **0.54 is the result**.
  Otherwise **0.52** the same way, then **0.50**.
  If 0.50 also fails, **no setting is accepted**: the lab records "not clean at 0.50", and you keep your Unit 7 submission for CARLA.
- **With the grip not identified**: race your Unit 7 submission as it is, with only `MAX_STEER` changed to your gain, twice, stopping at the first race that is not clean.
  Clean twice: that is the result; otherwise **no setting is accepted**, and you keep your Unit 7 submission.

The lab **never reports a value it did not race**: the result is always a setting that was clean on both of its races, or "no setting is accepted".

**The run budget**: R1 and R2, six probes, and at most six decision races: **14 runs**, about 2.5 hours at about 10 minutes per full race (the probes and the steady lap are shorter).
The **short form** skips R1 and the probes: the steering gain from R2, the grip not identified, and the no-estimate path, within **3 runs** (R2 and at most 2 races).
A host failure (the server exits, or the agent gets no first tick) is repeated and not counted: it says nothing about the agent.

Log every counted decision race in `RUNS`, in the order you raced them, and run the cell: it replays the procedure on your log, checks that each run is the one the procedure asked for (and that no run is logged after the procedure ended), and prints the grip to race with, the next run, the runs used and the result.

In [ ]:
LADDER = (0.54, 0.52, 0.50)  # MU_FRACTION, one grid step at a time
RACES_PER_SETTING = 2  # a setting is accepted only when it is clean on both of its races
STEER_DEG = 66.0  # racekit's measured steering gain, rounded
SHORT_FORM = False  # True: R2 only, no R1 and no probes

MY_GAIN_DEG = None  # your R2 gain (degrees), or None to ship 66
# Step 3: one line per probe from section 4, section: (qualifying rows, p99 or None, respawned)
PROBES = {}
# Step 4: your decision races, in the order you raced them: (setting, clean by section 3's rule), where setting
# is ("fraction", MU_FRACTION) with the grip identified, or ("unit-07",) on the no-estimate path.
RUNS = []


class NextRun(Exception):
    """The procedure needs a run that is not in the log yet."""


def grip_from_probes(probes):
    """CARLA's MU_G from the six probe lines, or None when the grip is not identified; and notes."""
    kinds = {s: verdict(*probes[s]) for s in range(SECTIONS)}
    notes = [f"section {s}: {kinds[s]}" + ("" if probes[s][1] is None else f", {probes[s][1]:.2f} m/s^2")
             for s in range(SECTIONS)]
    estimates = [probes[s][1] for s in range(SECTIONS) if kinds[s] == "estimate"]
    if not estimates:
        return None, notes + ["grip not identified: no probe slid, so the no-estimate path"]
    lowest = min(estimates)
    mu_g = tuple(round(lowest if kinds[s] == "no estimate" else probes[s][1], 2) for s in range(SECTIONS))
    return mu_g, notes + [f"MU_G = {mu_g} (a section with no estimate takes the lowest estimate, {lowest:.2f})"]


def replay(probes, runs, gain_deg=None, short_form=False):
    """Replay the procedure: returns (the next step or None, the notes, the result or None)."""
    notes, log = [], list(runs)
    steer = STEER_DEG if gain_deg is None else float(round(gain_deg))
    notes.append(f"MAX_STEER = math.radians({steer:.0f})")

    def race(label, setting):
        if not log:
            raise NextRun(f"{label}: {setting}")
        logged, clean = log.pop(0)
        if tuple(logged) != tuple(setting):
            raise ValueError(f"run {len(runs) - len(log)} is not the run the procedure asked for ({label})")
        notes.append(f"run {len(runs) - len(log):2d}, {label}: {'clean' if clean else 'NOT clean'}")
        return clean

    def finish(result):
        if log:
            raise ValueError(f"{len(log)} run(s) logged after the procedure ended; remove them or fix the log")
        notes.append(f"the result: {result}")
        return None, notes, result

    try:
        if short_form:
            mu_g = None
            notes.append("the short form: no probes, so the grip is not identified")
        else:
            missing = [s for s in range(SECTIONS) if s not in probes]
            if missing:
                raise NextRun(f"step 3, the probe of section {missing[0]} (GRIP[{missing[0]}] = 3.5)")
            mu_g, grip_notes = grip_from_probes(probes)
            notes.extend(grip_notes)
        if mu_g is None:
            setting = ("unit-07",)
            # all() stops at the first race that is not clean
            if all(race(f"no-estimate path, race {i + 1}", setting) for i in range(RACES_PER_SETTING)):
                return finish(f"your Unit 7 submission with MAX_STEER {steer:.0f} deg, clean on both races")
            return finish("no setting is accepted: keep your Unit 7 submission")
        for fraction in LADDER:
            setting = ("fraction", fraction)
            if all(race(f"MU_FRACTION {fraction}, race {i + 1}", setting) for i in range(RACES_PER_SETTING)):
                return finish(f"MU_FRACTION = {fraction}, MU_G = {mu_g}, MAX_STEER {steer:.0f} deg, clean on both races")
        return finish(f"not clean at {LADDER[-1]}: no setting is accepted; keep your Unit 7 submission")
    except NextRun as needed:
        return str(needed), notes, None


# Worked examples (made up, not recordings). Six probes: four slid, one never slid, one had too few rows.
EXAMPLE_PROBES = {0: (480, 5.61, False), 1: (900, 8.12, True), 2: (350, 8.40, True), 3: (12, None, True),
                  4: (410, 7.95, True), 5: (380, 8.66, True)}
example_mu_g, _ = grip_from_probes(EXAMPLE_PROBES)
assert example_mu_g == (5.61, 8.12, 8.40, 7.95, 7.95, 8.66)  # the lower bound kept, section 3 takes the lowest

# 0.54 clean twice: the result after two races
next_step, _, result = replay(EXAMPLE_PROBES, [(("fraction", 0.54), True)] * 2)
assert next_step is None and result.startswith("MU_FRACTION = 0.54")
# 0.54 fails at once, 0.52 fails on its second race, 0.50 is clean twice: the result is 0.50, after five races
ladder_runs = [(("fraction", 0.54), False), (("fraction", 0.52), True), (("fraction", 0.52), False),
               (("fraction", 0.5), True), (("fraction", 0.5), True)]
next_step, _, result = replay(EXAMPLE_PROBES, ladder_runs)
assert next_step is None and result.startswith("MU_FRACTION = 0.5,")
# every value fails: no setting is accepted, and a race logged after that is rejected
failing_runs = [(("fraction", 0.54), False), (("fraction", 0.52), False), (("fraction", 0.5), True),
                (("fraction", 0.5), False)]
next_step, _, result = replay(EXAMPLE_PROBES, failing_runs)
assert next_step is None and result.startswith("not clean at 0.5")
try:
    replay(EXAMPLE_PROBES, failing_runs + [(("fraction", 0.48), True)])
except ValueError as extra:
    assert "after the procedure ended" in str(extra)
else:
    raise AssertionError("an extra run was accepted")
# no probe slid: the no-estimate path; and an unfinished log names the next race
no_slide = {s: (400, 6.0, False) for s in range(SECTIONS)}
assert grip_from_probes(no_slide)[0] is None
next_step, _, result = replay(no_slide, [(("unit-07",), True)], gain_deg=64.6)
assert result is None and next_step == "no-estimate path, race 2: ('unit-07',)"
next_step, notes, result = replay(no_slide, [(("unit-07",), True), (("unit-07",), False)], gain_deg=64.6)
assert "MAX_STEER = math.radians(65)" in notes and result.startswith("no setting is accepted")

next_step, notes, result = replay(PROBES, RUNS, MY_GAIN_DEG, SHORT_FORM)
print("\n".join(notes))
recordings = 1 if SHORT_FORM else 2 + SECTIONS
budget = 7 if SHORT_FORM else recordings + len(LADDER) * RACES_PER_SETTING
print(f"runs: {recordings} recording and probe runs, plus {len(RUNS)} decision races logged so far; at most {budget} in all")
print(f"next: {next_step}" if next_step else "the procedure is complete")

## What to take away

- A car's numbers are **measured** on the car you race, not carried over from a model: your gain, your map, your grip.
- A fit is only as good as its rows: the race log excites the brake, the steady lap holds the steer still, and a probe is the only log that reaches the grip.
- A probe that never slid is a **lower bound**, and the lab calls it one; a section with too few rows has **no estimate**, and the lab does not invent one.
- The result is a setting raced clean **twice**, or no setting at all: a value that was never raced is never reported.

> **One last check: never commit your results.**
> This repository is public.
> Your recordings, their `.sections.txt` sidecars, screenshots and logs stay on your own machine, in the folder outside the repository
> that `ROAR_LAB_CSV` pointed at.
> Run `git status` in your clone before you commit anything: none of them should be listed.